In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Local Inference on GPU
Model page: https://huggingface.co/TinPhan2007/qwen-legal-lora

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/TinPhan2007/qwen-legal-lora)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [ ]:
!pip install -q -U transformers peft accelerate torchao
!pip install -q jupyter_black

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 86.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 81.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 78.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.1/95.1 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.4/268.4 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 111.8 MB/s eta 0:00:00


In [ ]:
!pip install rouge-score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=5c17942789630974c54e867ab05cac213e5f5164657d1fe9a43e4e0e54667728
  Stored in directory: /root/.cache/pip/wheels/44/af/da/5ffc433e2786f0b1a9c6f458d5fb8f611d8eb332387f18698f
Successfully built rouge-score


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

repo_id = "TinPhan2007/qwen-legal-lora-raumabk"
sub_folder = "qwen_legal_lora"

base_model_id = "Qwen/Qwen2.5-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(base_model_id, trust_remote_code=True)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

model = PeftModel.from_pretrained(
    base_model,
    repo_id,
    subfolder=sub_folder  
)
model.eval()

print("Đã tải xong mô hình!")

In [ ]:
cau_hoi = "Tôi đã đánh một người vì họ cố gắng xâm hại đến tài sản của tôi, nhưng tôi đã dùng vũ lực quá mức. Liệu tôi có phải bồi thường không?"

messages = [
    {
        "role": "system",
        "content": "Bạn là một trợ lý pháp lý chuyên nghiệp. Với mỗi câu trả lời, bạn BẮT BUỘC phải tìm và trích dẫn rõ ràng số điều luật, tên bộ luật liên quan và nội dung chi tiết của điều luật đó tại Việt Nam."
    },
    {
        "role": "user",
        "content": cau_hoi
    }
]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

model_inputs = tokenizer([text], return_tensors="pt").to("cuda")

with torch.no_grad():
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=512,
        temperature=0.7,
        top_p=0.9,
        do_sample=True
    )

generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

ket_qua = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
print(f"Câu hỏi: {cau_hoi}\n")
print(f"Trả lời:\n{ket_qua}")


Câu hỏi: Tôi đã đánh một người vì họ cố gắng xâm hại đến tài sản của tôi, nhưng tôi đã dùng vũ lực quá mức. Liệu tôi có phải bồi thường không?

Trả lời:
**Căn cứ pháp lý:** Điều 13 - Bộ luật Hình sự (Văn bản hợp nhất 2017)

**Nội dung quy định:**
Điều 13. Quyền chống lại người phạm tội 1. Người bị tấn công về thân thể, lợi dụng tình thế cấp thiết để bảo vệ mình hoặc người khác chống lại một kẻ đang chuẩn bị gây thiệt hại cho mình hoặc người khác có quyền sử dụng vũ lực nhẹ hoặc vũ lực mạnh. 2. Việc sử dụng vũ lực trong trường hợp không được quy định tại khoản 1 Điều này là tội phạm.

**Phân tích & Hướng dẫn:**
Căn cứ vào quy định nêu trên, đối với thắc mắc "Tôi đã đánh một người vì họ cố gắng xâm hại đến tài sản của tôi, nhưng tôi đã dùng vũ lực quá mức. Liệu tôi có phải bồi thường không?", vấn đề này được điều chỉnh và áp dụng trực tiếp theo các nguyên tắc, phạm vi của Điều 13 thuộc Bộ luật Hình sự (Văn bản hợp nhất 2017).


In [ ]:
import json
import torch
from tqdm import tqdm
from rouge_score import rouge_scorer


with open("/content/drive/MyDrive/test.json", "r", encoding="utf-8") as f:
    test_data = json.load(f)

scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

total_samples = len(test_data)
correct_citations = 0
correct_formats = 0
total_rouge_l = 0.0
output_details = []  

print(f"Đang chạy đánh giá trên toàn bộ {total_samples} mẫu test...")

for item in tqdm(test_data):
    law_id = item["law_id"]                        
    ground_truth = item["messages"][-1]["content"] 
    input_messages = item["messages"][:-1]         

    text = tokenizer.apply_chat_template(input_messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text], return_tensors="pt").to("cuda")

    with torch.no_grad():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=512,
            temperature=0.1,
            do_sample=False
        )

    generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)]
    predicted_text = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0] # Lấy chuỗi text đầu ra

    is_citation_correct = law_id.lower() in predicted_text.lower()
    if is_citation_correct:
        correct_citations += 1

    has_part1 = "**Căn cứ pháp lý:**" in predicted_text
    has_part2 = "**Nội dung quy định:**" in predicted_text
    has_part3 = "**Phân tích & Hướng dẫn:**" in predicted_text
    is_format_correct = has_part1 and has_part2 and has_part3
    if is_format_correct:
        correct_formats += 1

    scores = scorer.score(ground_truth, predicted_text)
    rouge_l_score = scores['rougeL'].fmeasure
    total_rouge_l += rouge_l_score

    output_details.append({
        "id": item.get("id", ""),
        "question": item.get("question", ""),
        "target_law_id": law_id,
        "ground_truth": ground_truth,
        "model_prediction": predicted_text,
        "metrics": {
            "citation_accuracy": is_citation_correct,
            "format_adherence": is_format_correct,
            "rouge_l": round(rouge_l_score * 100, 2)
        }
    })

summary_results = {
    "total_samples": total_samples,
    "overall_metrics": {
        "citation_accuracy_percentage": round((correct_citations / total_samples) * 100, 2),
        "format_adherence_percentage": round((correct_formats / total_samples) * 100, 2),
        "avg_rouge_l_percentage": round((total_rouge_l / total_samples) * 100, 2)
    },
    "detailed_results": output_details
}

with open("evaluation_results.json", "w", encoding="utf-8") as f:
    json.dump(summary_results, f, ensure_ascii=False, indent=4)

print("\n" + "="*40)
print(f"Tổng số mẫu test       : {total_samples}")
print(f"1. Citation Accuracy   : {summary_results['overall_metrics']['citation_accuracy_percentage']}%")
print(f"2. Format Adherence    : {summary_results['overall_metrics']['format_adherence_percentage']}%")
print(f"3. ROUGE-L Trung bình  : {summary_results['overall_metrics']['avg_rouge_l_percentage']}%")
print("="*40)
print("Đã xuất file kết quả chi tiết tại: evaluation_results.json")

In [ ]:
import json

samples_run = len(output_details)

if samples_run > 0:
    salvage_results = {
        "total_samples_tested": samples_run,
        "note": f"Đã ép dừng để cứu dữ liệu. Chạy được {samples_run}/484 câu.",
        "overall_metrics": {
            "citation_accuracy_percentage": round((correct_citations / samples_run) * 100, 2),
            "format_adherence_percentage": round((correct_formats / samples_run) * 100, 2),
            "avg_rouge_l_percentage": round((total_rouge_l / samples_run) * 100, 2)
        },
        "detailed_results": output_details
    }

    # Ghi file khẩn cấp
    with open("evaluation_results_partial.json", "w", encoding="utf-8") as f:
        json.dump(salvage_results, f, ensure_ascii=False, indent=4)

    print(f"ĐÃ CỨU THÀNH CÔNG {samples_run} MẪU!")
    print("Hãy tải ngay file 'evaluation_results_partial.json' ở cột bên trái về máy tính!")
else:
    print("Chưa có mẫu nào được ghi nhận.")


In [ ]:
import json
import torch
import os
from tqdm import tqdm
from rouge_score import rouge_scorer


old_results_path = "/content/drive/MyDrive/evaluation_results_partial.json"
test_set_path = "/content/drive/MyDrive/test.json"
final_output_path = "evaluation_results_final.json"


with open(test_set_path, "r", encoding="utf-8") as f:
    full_test_data = json.load(f)

old_output_details = []
if os.path.exists(old_results_path):
    with open(old_results_path, "r", encoding="utf-8") as f:
        old_data = json.load(f)
        if isinstance(old_data, dict) and "detailed_results" in old_data:
            old_output_details = old_data["detailed_results"]
        elif isinstance(old_data, list):
            old_output_details = old_data

remaining_test_data = full_test_data[len(old_output_details):]

print(f"Tổng số mẫu gốc            : {len(full_test_data)}")
print(f"Số mẫu lấy từ file cũ      : {len(old_output_details)}")
print(f"Số mẫu chạy tiếp sức       : {len(remaining_test_data)} mẫu.")


scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
new_output_details = []

if len(remaining_test_data) > 0:
    for item in tqdm(remaining_test_data, desc="Đang chạy nốt các mẫu còn thiếu"):
        law_id = item["law_id"]
        ground_truth = item["messages"][-1]["content"]
        input_messages = item["messages"][:-1]

        text = tokenizer.apply_chat_template(input_messages, tokenize=False, add_generation_prompt=True)
        model_inputs = tokenizer([text], return_tensors="pt").to("cuda")

        with torch.no_grad():
            generated_ids = model.generate(
                **model_inputs,
                max_new_tokens=512,
                temperature=0.1,
                do_sample=False
            )

        generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)]
        predicted_text = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

        # Tính toán metric
        is_citation_correct = law_id.lower() in predicted_text.lower()

        has_part1 = "**Căn cứ pháp lý:**" in predicted_text
        has_part2 = "**Nội dung quy định:**" in predicted_text
        has_part3 = "**Phân tích & Hướng dẫn:**" in predicted_text
        is_format_correct = has_part1 and has_part2 and has_part3

        scores = scorer.score(ground_truth, predicted_text)
        rouge_l_score = scores['rougeL'].fmeasure

        new_output_details.append({
            "id": item.get("id", ""),
            "question": item.get("question", ""),
            "target_law_id": law_id,
            "ground_truth": ground_truth,
            "model_prediction": predicted_text,
            "metrics": {
                "citation_accuracy": is_citation_correct,
                "format_adherence": is_format_correct,
                "rouge_l": round(rouge_l_score * 100, 2)
            }
        })


all_detailed_results = old_output_details + new_output_details
total_samples = len(all_detailed_results)

correct_citations = sum(1 for x in all_detailed_results if x.get("metrics", {}).get("citation_accuracy"))
correct_formats = sum(1 for x in all_detailed_results if x.get("metrics", {}).get("format_adherence"))
total_rouge_l = sum(x.get("metrics", {}).get("rouge_l", 0.0) for x in all_detailed_results)

summary_results = {
    "total_samples": total_samples,
    "overall_metrics": {
        "citation_accuracy_percentage": round((correct_citations / total_samples) * 100, 2),
        "format_adherence_percentage": round((correct_formats / total_samples) * 100, 2),
        "avg_rouge_l_percentage": round(total_rouge_l / total_samples, 2)
    },
    "detailed_results": all_detailed_results
}

with open(final_output_path, "w", encoding="utf-8") as f:
    json.dump(summary_results, f, ensure_ascii=False, indent=4)

print("\n" + "="*45)
print(f"Tổng số mẫu toàn tập test  : {total_samples}")
print(f"1. Citation Accuracy       : {summary_results['overall_metrics']['citation_accuracy_percentage']}%")
print(f"2. Format Adherence        : {summary_results['overall_metrics']['format_adherence_percentage']}%")
print(f"3. ROUGE-L Trung bình      : {summary_results['overall_metrics']['avg_rouge_l_percentage']}%")
print("="*45)
print(f" File báo cáo: {final_output_path}")

Tổng số mẫu gốc            : 484
Số mẫu lấy từ file cũ      : 458
Số mẫu chạy tiếp sức       : 26 mẫu.


Đang chạy nốt các mẫu còn thiếu: 100%|██████████| 26/26 [14:02<00:00, 32.40s/it]


Tổng số mẫu toàn tập test  : 484
1. Citation Accuracy       : 8.88%
2. Format Adherence        : 84.71%
3. ROUGE-L Trung bình      : 62.2%
 File báo cáo: evaluation_results_final.json
